In [1]:
# script to add the activity predictions from PARM to mpac predictions from mpac promtoer seqlet calling 112125
# for initial acitvity correlations of promoters with PARM

In [1]:
# import packages
import pandas as pd
import numpy as np
import os
import seaborn as sns
from scipy import stats

In [2]:
# open mpac predictions
mpac_preds = pd.read_csv('../processed_data/mpac.250bp.promoter.preds.with.annotations.tsv', sep = '\t')

In [3]:
# open all hepg2 parm predictions
# change to directory
os.chdir('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/PARM/preds_out/all_gencode_v44_activity_preds_hepg2')
hepg2_parm_activity = pd.concat([pd.read_csv(i, sep = '\t') for i in os.listdir()])
# add ENSG ID as column for building dictionary
hepg2_parm_activity.loc[:, 'gene_id'] = [i.split('_')[0] for i in hepg2_parm_activity['header']]
# build dictionary
hepg2_parm_act_dict = dict(zip(hepg2_parm_activity['gene_id'], hepg2_parm_activity['prediction_HepG2']))

In [4]:
# open all k562 parm predictions
# change to directory
os.chdir('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/PARM/preds_out/all_gencode_v44_activity_preds_k562')
k562_parm_activity = pd.concat([pd.read_csv(i, sep = '\t') for i in os.listdir()])
# add ENSG ID as column for building dictionary
k562_parm_activity.loc[:, 'gene_id'] = [i.split('_')[0] for i in k562_parm_activity['header']]
# build dictionary
k562_parm_act_dict = dict(zip(k562_parm_activity['gene_id'], k562_parm_activity['prediction_K562']))

In [5]:
# iterate through gene_ids and pull parm activity predictions
mpac_preds.loc[:, 'hepg2_parm_act'] = [hepg2_parm_act_dict.get(i) for i in mpac_preds['gene_id']]
mpac_preds.loc[:, 'k562_parm_act'] = [k562_parm_act_dict.get(i) for i in mpac_preds['gene_id']]

In [22]:
# save to disk
mpac_preds.to_csv('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/processed_data/mpac.250bp.promoter.preds.with.annotations.parm.act.tsv', sep = '\t', index = False)

In [ ]:
# pause here 11/23/25: 5:10PM and add promoterAI predictions, PARM saturation mutagenesis predictions will finsih 11/24 AM

In [5]:
# open current version of the df
promoter_preds_df = pd.read_csv('../processed_data/mpac.250bp.promoter.preds.with.annotations.parm.act.promoterAI.tsv', sep = '\t')

In [6]:
promoter_preds_df.head()

,chrom,pos,id,ref,alt,k562_ref_pred,k562_alt_pred,k562_skew_pred,hepg2_ref_pred,hepg2_alt_pred,...,hepg2_emVar,sknsh_emVar,emVar_any,emVar_all,k562_rna_tpm,hepg2_rna_tpm,sknsh_rna_tpm,hepg2_parm_act,k562_parm_act,promoterAI_score
0,chr1,65169,ENSG00000186092.7_ENST00000641515.2_OR4F5..chr...,A,C,0.368796,0.382817,0.014021,0.367238,0.361899,...,0,0,0,0,0.0,0.0,0.0,0.667815,0.166738,0.005
1,chr1,65169,ENSG00000186092.7_ENST00000641515.2_OR4F5..chr...,A,G,0.368796,0.373723,0.004927,0.367238,0.363106,...,0,0,0,0,0.0,0.0,0.0,0.667815,0.166738,-0.002
2,chr1,65169,ENSG00000186092.7_ENST00000641515.2_OR4F5..chr...,A,T,0.368796,0.355564,-0.013233,0.367238,0.344585,...,0,0,0,0,0.0,0.0,0.0,0.667815,0.166738,-0.003
3,chr1,65170,ENSG00000186092.7_ENST00000641515.2_OR4F5..chr...,G,A,0.287499,0.341029,0.053531,0.352586,0.411632,...,0,0,0,0,0.0,0.0,0.0,0.667815,0.166738,0.010
4,chr1,65170,ENSG00000186092.7_ENST00000641515.2_OR4F5..chr...,G,C,0.287499,0.300818,0.013319,0.352586,0.341317,...,0,0,0,0,0.0,0.0,0.0,0.667815,0.166738,0.002


In [2]:
# and we're back (11/24/25). add all sat mut parm predictions to the dataframe
k562_parm_preds = pd.read_csv('../PARM/processed_data/k562_parm_preds.tsv.gz', sep = '\t')
hepg2_parm_preds = pd.read_csv('../PARM/processed_data/hepg2_parm_preds.tsv.gz', sep = '\t')

In [3]:
# make dictionaries of the merge_id and skew for adding to df
k562_parm_pred_dict = dict(zip(k562_parm_preds['merge_id'], k562_parm_preds['skew']))
hepg2_parm_pred_dict = dict(zip(hepg2_parm_preds['merge_id'], hepg2_parm_preds['skew']))

In [7]:
# add predictions to the full df
promoter_preds_df.loc[:, 'parm_k562_pred'] = [-1 * k562_parm_pred_dict.get(i) for i in promoter_preds_df['merge_id']] # multiply by -1 because PARM does Ref - Alt
promoter_preds_df.loc[:, 'parm_hepg2_pred'] = [-1 * hepg2_parm_pred_dict.get(i) for i in promoter_preds_df['merge_id']]
# add the mean of the two cell types
promoter_preds_df.loc[:, 'parm_mean_pred'] = [np.mean([k,h]) for k,h in zip(promoter_preds_df['parm_k562_pred'], promoter_preds_df['parm_hepg2_pred'])]

In [8]:
# get the 95th percentile for an approximation of the emVar threshold
all_parm_skews = []
for k, h in zip(promoter_preds_df['parm_k562_pred'], promoter_preds_df['parm_hepg2_pred']):
    all_parm_skews.append(abs(k))
    all_parm_skews.append(abs(h))

parm_95 = np.quantile(all_parm_skews, 0.95)

In [9]:
parm_95

np.float64(0.16889143)

In [43]:
# annotate df
promoter_preds_df.loc[:, 'parm_k562_emVar'] = [1 if abs(i) > parm_95 else 0 for i in promoter_preds_df['parm_k562_pred']]
promoter_preds_df.loc[:, 'parm_hepg2_emVar'] = [1 if abs(i) > parm_95 else 0 for i in promoter_preds_df['parm_hepg2_pred']]
promoter_preds_df.loc[:, 'parm_emVar_any'] = [1 if sum([k,h]) > 0 else 0 for k,h in zip(promoter_preds_df['parm_k562_emVar'],
                                                                                        promoter_preds_df['parm_hepg2_emVar'])]

In [47]:
# save df to disk
promoter_preds_df.to_csv('../processed_data/mpac.250bp.promoter.preds.with.annotations.parm.act.promoterAI.parm.sat.mut.tsv', sep = '\t', index = False)